# Hi , I am just learning , if there are any suggestions or topics that I can learn to improve please let me know

In [ ]:
# IMPORTING LIBRARIES
import pandas as pd
import numpy as np
import tensorflow as tf 
import os
import zipfile
from tensorflow import keras
from keras import layers

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vbookshelf/rice-leaf-diseases")

print("Path to dataset files:", path)

In [ ]:
data_dir='/kaggle/input/rice-leaf-diseases/rice_leaf_diseases/'
disease_names=[
    folder for folder in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir,folder))
]
print('Disease names:',disease_names)

In [ ]:
for disease in disease_names:
    folder_path = os.path.join(data_dir, disease)
    num_images = len([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.png', '.jpeg'))
    ])
    print(f"{disease}: {num_images} images")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

plt.figure(figsize=(12,4))

for i, disease in enumerate(disease_names):
    folder_path = os.path.join(data_dir, disease)
    images = [
        img for img in os.listdir(folder_path)
        if img.lower().endswith(('.jpg', '.png', '.jpeg'))
    ]
    
    img_path = os.path.join(folder_path, random.choice(images))
    img = Image.open(img_path)

    plt.subplot(1, len(disease_names), i+1)
    plt.imshow(img)
    plt.title(disease)
    plt.axis("off")

plt.show()


In [ ]:
###  splitting data 
train_ds=tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    image_size=(224,224),
    subset='training',validation_split=0.1,
    label_mode='categorical',batch_size=4
    ,seed=42,shuffle=True
)
validation_ds=tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,image_size=(224,224),subset='validation',
    validation_split=0.1,label_mode='categorical',batch_size=4,seed=42,shuffle=True
)

In [ ]:
early_stop=tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5, restore_best_weights=True
)
lr_schedule=tf.keras.callbacks.ReduceLROnPlateau(
    monitor= 'val_loss', factor=0.3,patience=3,min_lr=1e-6,verbose=1
)

data_augmentation=tf.keras.Sequential([
    layers.RandomFlip('Horizontal'),
    layers.RandomRotation(0.20),
    layers.RandomZoom(0.3)
])

In [ ]:
base_model=tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,weights='imagenet'
)

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base_model.trainable=False

inputs=tf.keras.Input(shape=(224,224,3))
x = preprocess_input(inputs)

x=data_augmentation(x)
x=base_model(x,training = False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dense(64,activation='relu')(x)
x=tf.keras.layers.Dropout(0.4)(x)

output=tf.keras.layers.Dense(3,activation='softmax')(x)
tf_model=tf.keras.Model(inputs,output)

In [ ]:
tf_model.summary()

In [ ]:
tf_model.compile(optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history2=tf_model.fit(train_ds,validation_data=validation_ds,epochs=30,verbose =1,callbacks=[early_stop,lr_schedule])

In [ ]:
val_loss, val_accuracy = tf_model.evaluate(validation_ds)
print(f"Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history2.history['accuracy'], label='Train')
plt.plot(history2.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history2.history['loss'], label='Train')
plt.plot(history2.history['val_loss'], label='Validation')
plt.title('Loss')
plt.legend()

plt.show()